# 1. Setup and Paths

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import urllib.request

import pandas as pd

In [5]:
current_dir = Path.cwd().resolve()
project_name = 'ride-hailing-demand-fleet-allocation'

if current_dir.name == project_name:
    project_root = current_dir
elif current_dir.name == 'notebooks' and current_dir.parent.name == project_name:
    project_root = current_dir.parent
else:
    project_root = current_dir / project_name

if not project_root.exists():
    raise FileNotFoundError(
        f'Project folder not found: {project_root}'
    )

raw_dir = project_root / 'data' / 'raw'
manifest_dir = project_root / 'data' / 'manifests'

raw_dir.mkdir(parents=True, exist_ok=True)
manifest_dir.mkdir(parents=True, exist_ok=True)

print(f'Current directory : {current_dir}')
print(f'Project root      : {project_root}')
print(f'Raw directory     : {raw_dir}')
print(f'Manifest directory: {manifest_dir}')

# 2. Source Manifest

In [6]:
trip_base_url = ('https://d37ci6vzurychx.cloudfront.net/trip-data')
lookup_url = ('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')
source_files = {}

for month in range(1, 13):
    month_text = f'{month:02d}'
    filename = (f'fhvhv_tripdata_2024-{month_text}.parquet')
    url = f'{trip_base_url}/{filename}'
    source_files[filename] = url

source_files['taxi_zone_lookup.csv'] = lookup_url
print(f'Expected files: {len(source_files)}')

for filename in source_files:
    print(filename)

Expected files: 13
fhvhv_tripdata_2024-01.parquet
fhvhv_tripdata_2024-02.parquet
fhvhv_tripdata_2024-03.parquet
fhvhv_tripdata_2024-04.parquet
fhvhv_tripdata_2024-05.parquet
fhvhv_tripdata_2024-06.parquet
fhvhv_tripdata_2024-07.parquet
fhvhv_tripdata_2024-08.parquet
fhvhv_tripdata_2024-09.parquet
fhvhv_tripdata_2024-10.parquet
fhvhv_tripdata_2024-11.parquet
fhvhv_tripdata_2024-12.parquet
taxi_zone_lookup.csv


# 3. Download Raw File

In [7]:
download_results = []

for filename, url in source_files.items():
    file_path = raw_dir / filename
    if (file_path.exists() and file_path.stat().st_size > 0):
        status = 'existing'
        print(f'[SKIP] {filename}')
    else:
        print(f'[DOWNLOAD] {filename}')
        urllib.request.urlretrieve(url, file_path)
        status = 'downloaded'
        print(f'[DONE] {filename}')

    download_results.append([filename, url, status])

[SKIP] fhvhv_tripdata_2024-01.parquet
[SKIP] fhvhv_tripdata_2024-02.parquet
[SKIP] fhvhv_tripdata_2024-03.parquet
[SKIP] fhvhv_tripdata_2024-04.parquet
[SKIP] fhvhv_tripdata_2024-05.parquet
[SKIP] fhvhv_tripdata_2024-06.parquet
[SKIP] fhvhv_tripdata_2024-07.parquet
[SKIP] fhvhv_tripdata_2024-08.parquet
[SKIP] fhvhv_tripdata_2024-09.parquet
[SKIP] fhvhv_tripdata_2024-10.parquet
[SKIP] fhvhv_tripdata_2024-11.parquet
[SKIP] fhvhv_tripdata_2024-12.parquet
[SKIP] taxi_zone_lookup.csv


In [9]:
manifest = pd.DataFrame(
    download_results,
    columns=[
        'filename',
        'source_url',
        'download_status'
    ]
)

manifest

,filename,source_url,download_status
0,fhvhv_tripdata_2024-01.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
1,fhvhv_tripdata_2024-02.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
2,fhvhv_tripdata_2024-03.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
3,fhvhv_tripdata_2024-04.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
4,fhvhv_tripdata_2024-05.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
5,fhvhv_tripdata_2024-06.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
6,fhvhv_tripdata_2024-07.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
7,fhvhv_tripdata_2024-08.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
8,fhvhv_tripdata_2024-09.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing
9,fhvhv_tripdata_2024-10.parquet,https://d37ci6vzurychx.cloudfront.net/trip-dat...,existing


# 4. Verify File Inventory

In [12]:
exists_results = []
size_bytes_results = []
size_mb_results = []

for filename in manifest['filename']:
    file_path = raw_dir / filename
    if file_path.exists():
        file_exists = True
        file_size_bytes = file_path.stat().st_size
    else:
        file_exists = False
        file_size_bytes = 0
    file_size_mb = round(file_size_bytes / (1024 ** 2), 2)

    exists_results.append(file_exists)
    size_bytes_results.append(file_size_bytes)
    size_mb_results.append(file_size_mb)

manifest['exists'] = exists_results
manifest['file_size_bytes'] = size_bytes_results
manifest['file_size_mb'] = size_mb_results

manifest[['filename', 'exists', 'file_size_mb']]

,filename,exists,file_size_mb
0,fhvhv_tripdata_2024-01.parquet,True,450.86
1,fhvhv_tripdata_2024-02.parquet,True,441.05
2,fhvhv_tripdata_2024-03.parquet,True,484.41
3,fhvhv_tripdata_2024-04.parquet,True,454.06
4,fhvhv_tripdata_2024-05.parquet,True,475.48
5,fhvhv_tripdata_2024-06.parquet,True,461.98
6,fhvhv_tripdata_2024-07.parquet,True,446.74
7,fhvhv_tripdata_2024-08.parquet,True,444.40
8,fhvhv_tripdata_2024-09.parquet,True,444.86
9,fhvhv_tripdata_2024-10.parquet,True,462.63


# 5. Calculate Checksums

In [13]:
checksum_results = []

for filename in manifest['filename']:
    file_path = raw_dir / filename
    print(f'[HASH] {filename}')
    if file_path.exists():
        with open(file_path, 'rb') as file:
            file_hash = hashlib.file_digest(
                file,
                'sha256'
            ).hexdigest()
    else:
        file_hash = ''

    checksum_results.append(file_hash)

[HASH] fhvhv_tripdata_2024-01.parquet
[HASH] fhvhv_tripdata_2024-02.parquet
[HASH] fhvhv_tripdata_2024-03.parquet
[HASH] fhvhv_tripdata_2024-04.parquet
[HASH] fhvhv_tripdata_2024-05.parquet
[HASH] fhvhv_tripdata_2024-06.parquet
[HASH] fhvhv_tripdata_2024-07.parquet
[HASH] fhvhv_tripdata_2024-08.parquet
[HASH] fhvhv_tripdata_2024-09.parquet
[HASH] fhvhv_tripdata_2024-10.parquet
[HASH] fhvhv_tripdata_2024-11.parquet
[HASH] fhvhv_tripdata_2024-12.parquet
[HASH] taxi_zone_lookup.csv


In [14]:
manifest['sha256'] = checksum_results

manifest[['filename', 'file_size_mb', 'sha256']]

,filename,file_size_mb,sha256
0,fhvhv_tripdata_2024-01.parquet,450.86,9897de352aa52cea36b70348cc6721b8d4494327ce39c8...
1,fhvhv_tripdata_2024-02.parquet,441.05,54bafa7bcf3f06e7833fcbccf7ef158eb793a72ee1c4bb...
2,fhvhv_tripdata_2024-03.parquet,484.41,ff25b93f890f889637963b749d0839ec5a15f075d23ea5...
3,fhvhv_tripdata_2024-04.parquet,454.06,1493f3edd0486bf6c3f949cb5a66ec49e9d04f7ad6e94f...
4,fhvhv_tripdata_2024-05.parquet,475.48,152db11cbc95a796fea0b9cb26184b3c4cfad181fa9d74...
5,fhvhv_tripdata_2024-06.parquet,461.98,4772ee1ab01e639a5db74169b49049f74f0f46a0c925fd...
6,fhvhv_tripdata_2024-07.parquet,446.74,3a511a8df72441d72b32f6ce9829e29f5d61666e37f1b4...
7,fhvhv_tripdata_2024-08.parquet,444.40,99fd2913d2c8935e4c1d166f229290a90971b6564c490a...
8,fhvhv_tripdata_2024-09.parquet,444.86,d2d26d608a775555128a2b435d375f5d25d3d8125b688c...
9,fhvhv_tripdata_2024-10.parquet,462.63,8ee625798a6543148c3a3e33df5e2cf44acb17af904dbf...


# 6. Save Acquisition Manifest

In [15]:
checked_at = datetime.now(timezone.utc).isoformat()
manifest['checked_at_utc'] = checked_at
manifest_path = (manifest_dir/'acquisition_manifest_2024.csv')
manifest.to_csv(manifest_path, index=False)

print(f'Manifest saved to: {manifest_path}')

Manifest saved to: D:\My Journey\Data Project\CLAUDE\ride-hailing-demand-fleet-allocation\data\manifests\acquisition_manifest_2024.csv


# 7. Final Validation

In [17]:
expected_files = len(source_files)
manifest_rows = len(manifest)

existing_files = manifest['exists'].sum()
non_empty_files = (manifest['file_size_bytes'] > 0).sum()
valid_checksums = (manifest['sha256'].str.len() == 64).sum()

print(f'Expected files  : {expected_files}')
print(f'Manifest rows   : {manifest_rows}')
print(f'Existing files  : {existing_files}')
print(f'Non-empty files : {non_empty_files}')
print(f'Valid checksums : {valid_checksums}')

Expected files  : 13
Manifest rows   : 13
Existing files  : 13
Non-empty files : 13
Valid checksums : 13
